# 配置 workflow 的畫面事件

這份教材只看一件事：如何用 `events_schema` 指定畫面要接收哪些流程 label 與結構化欄位。它承接上一章的 stage event，不討論對話記憶、檢索品質或工具呼叫。

## 在 Colab 準備環境

如果你是在 Colab 開啟，先安裝 SDK；如果已經在專案環境，可以直接跳過。

In [ ]:
!pip install -q "git+https://github.com/R300-AI/Agentic-SDK.git"

## 填入模型連線設定

這章需要一個支援 OpenAI-compatible chat completion 的模型，因為 Perceive 與 Plan 會產生 JSON。請填入自己的端點與模型名稱。

In [ ]:
from agentic_sdk import Workflow
from agentic_sdk.modules import GenerativeAction, KeywordRetrieve, NextStepPlan, TextPerceive

API_KEY = '<API_KEY>'
BASE_URL = '<OpenAI-compatible BASE_URL>'
MODEL = '<MODEL>'

## 選擇要投影到畫面的事件

不傳 `events_schema` 時，SDK 會送出完整預設事件。當畫面只需要簡短、穩定的推論摘要時，傳入 schema 來縮限事件：每個 module 的 `label` 是畫面階段文字；`fields` 是完整 JSON 值送出後才會發生的 dot path。

下表逐一列出內建模組目前會產生的 fields。`fields` 只控制 `structured_field` 事件，不影響 workflow 執行；沒有 JSON fields 的節點仍會發送 stage event，LLM Action 也仍會發送 `token_delta`。

| 節點 | 預設 label | 模組 | field | 型別／可能值 | 說明 |
| --- | --- | --- | --- | --- | --- |
| `perceive` | 理解輸入 | `TextPerceive`／`TextImagePerceive` | `intent` | string，例如 `general` 或模型給的 snake_case 意圖 | 對使用者輸入的分類，放在 `perceived_intent` payload。 |
| `perceive` | 理解輸入 | `TextPerceive`／`TextImagePerceive` | `summary` | string | 可顯示的輸入摘要；模型沒有給時 SDK 以原始使用者訊息作 fallback。 |
| `perceive` | 理解輸入 | `TextPerceive`／`TextImagePerceive` | `details` | object | 模型補充的理解資料；選擇 `details` 會收到完整 object。 |
| `perceive` | 理解輸入 | `TextPerceive`／`TextImagePerceive` | `details.<key>` | 任意完整 JSON value | `details` 的每個模型提供 key 都可能出現；例如 `details.next_step`。使用 `"*"` 時 SDK 會逐一送出已完成的巢狀 key，不能將它當成固定欄位集合。 |
| `plan` | 判斷工具順序 | `NextStepPlan` | `thought` | string | 為畫面準備的決策摘要，不是要求模型輸出完整隱藏推理。 |
| `plan` | 判斷工具順序 | `NextStepPlan` | `next_module` | `retrieve` 或 `action` | 下一個要執行的 workflow 節點；不合法的模型值會由 SDK fallback 成 `action`。 |
| `retrieve` | 整理相關來源 | `KeywordRetrieve`、`SemanticRetrieve`、`PassThroughRetrieve` | 無 | - | 內建 Retrieve 不產生 `structured_field`；查到的資料寫入 workflow context，仍會有 stage event。 |
| `action` | 準備輸出回覆 | `DirectAnswerAction`、`GenerativeAction`、`ToolCallAction` | 無 | - | 內建 Action 不產生 `structured_field`。`GenerativeAction` 的可見文字以 `token_delta` 送出，最終結果在 `final_message`。 |
| `reflect` | 檢查回覆 | `ResponseCheckReflect` | `verdict` | `pass` 或 `fail` | 回覆檢查結論；`fail` 時依 `on_failure` 決定結束或回到 Plan。 |
| `reflect` | 檢查回覆 | `ResponseCheckReflect` | `reason` | string | 模型給出的檢查理由。 |
| `reflect` | 檢查回覆 | `ResponseCheckReflect` | `suggestion` | string | 模型給出的修正建議；空字串代表沒有建議。 |
| `reflect` | 檢查回覆 | `EvidenceCheckReflect` | 無 | - | 此內建 Reflect 以程式規則驗證 evidence，結果在 `reflect_verdict` payload／reflection metadata，不會發送 `structured_field`。 |

以上是內建模組的固定欄位。完整的「所有可能」欄位集合不是封閉的：自訂 Perceive、Plan、Reflect，或模型在 `details` 內新增 key，都可以產生更多 JSON path；設定 `fields: ["*"]` 會讓 SDK 送出每個完整 JSON value（包含巢狀 object 與 array value）。要穩定顯示 UI，請明確列出已審核的 path，例如 `summary`、`details.next_step`、`thought`、`next_module`、`verdict`，不要把 chain-of-thought 或未審核的任意模型文字放進 `fields`。

In [ ]:
events_schema = {
    'perceive': {
        'label': '理解輸入',
        'fields': ['summary', 'details.next_step'],
    },
    'plan': {
        'label': '判斷工具順序',
        'fields': ['thought', 'next_module'],
    },
    'retrieve': {'label': '整理相關來源'},
    'action': {'label': '準備輸出回覆'},
}

## 把 schema 放進 workflow

`events_schema` 是唯一的事件設定入口。沒有列出的 module 仍可正常執行，但不會送出 stage 或 structured-field event；LLM token delta 則仍可供需要完整串流的應用層處理。

In [ ]:
workflow = Workflow(
    events_schema=events_schema,
    perceive=TextPerceive(
        api_key=API_KEY,
        base_url=BASE_URL,
        model=MODEL,
        options=[{'name': 'next_step', 'description': 'The recommended next workflow step.'}],
    ),
    plan=NextStepPlan(api_key=API_KEY, base_url=BASE_URL, model=MODEL),
    retrieve=KeywordRetrieve(items=[{
        'keywords': ['sdk'],
        'content': 'Agentic SDK 會將 workflow 事件提供給應用程式。',
    }]),
    action=GenerativeAction(api_key=API_KEY, base_url=BASE_URL, model=MODEL),
)

## 等節點完成後一次顯示監測欄位

`structured_field` 的 `phase` 固定是 `field`，`status` 是 `completed`：它表示**該 field 的 JSON value 已完整**，不是整個 module 已完成。SDK 對同一 module visit 與 field path 只送一次，因此不會因 token 串流重複發送同一 field。

SDK 會在同一節點的 `stage` `phase == "finish"` 事件附上 `fields` list：其中是這次節點已完成、且符合 `events_schema` 的 `{ "field": ..., "value": ... }`。明確列出的 fields 會依 schema 的順序輸出；`"*"` 則列出該次實際完成的所有 path。畫面不必自行暫存。

In [ ]:
def on_event(event):
    if event['type'] == 'stage' and event['phase'] == 'start':
        print(f"\n【{event['label']}】")
    elif event['type'] == 'stage' and event['phase'] == 'finish':
        for item in event['fields']:
            print(f"{item['field']}：{item['value']}")

## 執行並確認節點結束時才顯示摘要

`summary`、`details.next_step`、`thought` 與 `next_module` 都會在對應節點的 `stage.finish` 事件中，以 schema 排序的 `fields` list 一次提供。最後回覆仍從 `result.final_message` 取得。

In [ ]:
result = workflow.run('請介紹 Agentic SDK 的事件顯示方式。', event_callback=on_event)
print('\n最終回覆：', result.final_message)